# Install dependencies

In [ ]:
!sudo apt-get install zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama fastapi uvicorn[standard]

In [ ]:
!sudo mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | sudo tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | sudo tee /etc/apt/sources.list.d/cloudflared.list
!sudo apt-get update -qq && sudo apt-get install -y cloudflared
!cloudflared --version
print("✅ cloudflared installed")
!sudo cloudflared service install YOUR_CLOUDFLARETUNNEL_TOKEN

#Qwen

In [ ]:
!nohup ollama serve > ollamaserve.log 2>&1 &
!ollama run qwen3.5:4b
# when ollama ask you enter in terminal to chat, terminate this cell and then proceed to server web API

#Web API

In [15]:
#write the script to api.py
api = '''from fastapi import FastAPI
from ollama import chat
app = FastAPI()
history = []

instructions = 'following are instructions you must strickly follow and keep in mind, dont answer to this msg. you are Usman, a problem solver who built an auto-chat system. the system automatically detects new message and respond to them. 1) in order to clear history, ask user to type clear. 2) you have to keep the answers concise, jolly, friendly and meaningful and only one liner. 3) in the messages, the upper most line might be garbade so ignore that as they might be the previous message that you sent. one way to detect it is by checking if it starts with keyword "you" which means it is me so dont consider it ignore 1-2 lines following it. 4) the answer should not exceed more than one line. 5) you might be in group chat so you will have to keep with different people, one way is to see if the message contains a name, then check rule 3 and then reply to that person with his history in mind. make sure to say his name so it is clear who you are talking too. 6) if you see that the last message mostly coorelates with the last text then do not reply to that msg, instead always say this "How can i help you?" in this case and just this. now proceed to the other messages. I repeat, following these instructions is must!' 
history.append({'role': 'user', 'content': instructions})

# Define a route for the root URL
@app.get("/")
def read_root():
    return "hello world";

# Define a route for the root URL
@app.get("/prompt/{msg}")
def read_msg(msg):
    if "clear" in msg:
        history.clear()
        history.append({'role': 'user', 'content': instructions})
        return {"reply": "history cleared"};
    history.append({'role': 'user', 'content': msg})
    response = chat(
      model='qwen3.5:4b',
      messages=history,
      think=False,
      stream=False,
    )
    return {"reply": response.message.content};
'''

with open('/content/api.py', 'w') as f:
    f.write(api)

In [16]:
#https://pytutorial.com/python-api-development-guide-for-beginners/
#https://docs.ollama.com/capabilities/thinking
import os
os.system("pkill -f cloudflared")
os.system("pkill -f ollama")
os.system("pkill -f uvicorn")
!nohup cloudflared tunnel run --token YOUR_CLOUDFLARETUNNEL_TOKEN > /dev/null 2>&1 &
!nohup ollama serve > ollamaserve.log 2>&1 &
!nohup uvicorn api:app --reload > api.log 2>&1 &